# D.R.O.N.A. — Retrieval Quality Benchmark (C1)

Measures Contribution C1: retrieval quality of the hybrid ChromaDB retriever.

**Metrics:**
- MRR@10 (Mean Reciprocal Rank)
- nDCG@10 (normalised Discounted Cumulative Gain)
- Nepal citation fraction per query
- Latency distribution

**Prerequisites:**
- ChromaDB populated: `python scripts/ingest_data.py`
- Relevance judgements file: `data/evaluation/relevance_judgements.json`
  (or run with synthetic judgements for smoke testing)

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from drona.utils.settings import settings
from drona.utils.logging import setup_logging
setup_logging('WARNING')

from drona.advising.retriever import Retriever
from drona.contracts import DataTier

print('Initialising retriever...')
retriever = Retriever()
print('Retriever ready.')

## 1. Load or Generate Test Queries

In [ ]:
judgements_path = Path('../data/evaluation/relevance_judgements.json')

if judgements_path.exists():
    with open(judgements_path) as f:
        judgements = json.load(f)
    print(f'Loaded {len(judgements)} queries from {judgements_path}')
else:
    # Synthetic fallback — used for CI / smoke testing
    print('relevance_judgements.json not found — using synthetic test queries.')
    print('For real evaluation, create data/evaluation/relevance_judgements.json')
    judgements = [
        {
            'query': 'Python developer jobs in Nepal',
            'relevant_ids': [],  # unknown for synthetic
            'label': 'nepal_python'
        },
        {
            'query': 'Data science career pathway BSc Computing',
            'relevant_ids': [],
            'label': 'data_science'
        },
        {
            'query': 'Software engineering job Kathmandu entry level',
            'relevant_ids': [],
            'label': 'swe_kathmandu'
        },
        {
            'query': 'Machine learning opportunities Nepal fintech',
            'relevant_ids': [],
            'label': 'ml_fintech'
        },
        {
            'query': 'Cybersecurity analyst skills Nepal',
            'relevant_ids': [],
            'label': 'cybersec'
        },
    ]

print(f'Test set size: {len(judgements)} queries')

## 2. Run Retrieval Benchmark

In [ ]:
def reciprocal_rank(retrieved_ids, relevant_ids, k=10):
    """Compute reciprocal rank of first relevant result in top-k."""
    if not relevant_ids:
        return None  # cannot evaluate without ground truth
    for i, doc_id in enumerate(retrieved_ids[:k]):
        if doc_id in relevant_ids:
            return 1.0 / (i + 1)
    return 0.0

def ndcg(retrieved_ids, relevant_ids, k=10):
    """Compute nDCG@k with binary relevance."""
    if not relevant_ids:
        return None
    gains = [1.0 if doc_id in relevant_ids else 0.0 for doc_id in retrieved_ids[:k]]
    dcg = sum(g / np.log2(i + 2) for i, g in enumerate(gains))
    ideal = sum(1.0 / np.log2(i + 2) for i in range(min(len(relevant_ids), k)))
    return dcg / ideal if ideal > 0 else 0.0

results = []
for item in judgements:
    query = item['query']
    relevant_ids = set(item.get('relevant_ids', []))
    
    t0 = time.perf_counter()
    docs = retriever.retrieve_raw(query, top_k=10)
    latency_ms = (time.perf_counter() - t0) * 1000
    
    retrieved_ids = [d.id for d in docs]
    nepal_count = sum(1 for d in docs if d.metadata.get('tier') == 'nepal')
    
    rr = reciprocal_rank(retrieved_ids, relevant_ids)
    ndcg_score = ndcg(retrieved_ids, relevant_ids)
    
    results.append({
        'query': query[:50],
        'label': item.get('label', ''),
        'n_retrieved': len(docs),
        'nepal_fraction': nepal_count / len(docs) if docs else 0,
        'latency_ms': latency_ms,
        'rr': rr,
        'ndcg': ndcg_score,
        'top3_ids': retrieved_ids[:3],
    })
    print(f'  [{item.get("label","?"):20s}] {len(docs):2d} docs | Nepal={nepal_count}/{len(docs)} | {latency_ms:.0f}ms')

df = pd.DataFrame(results)
print(f'\n{len(df)} queries evaluated.')

## 3. Metrics Summary

In [ ]:
evaluable = df[df['rr'].notna()]

if len(evaluable) > 0:
    mrr = evaluable['rr'].mean()
    mean_ndcg = evaluable['ndcg'].mean()
    print(f'MRR@10:         {mrr:.4f}  (target ≥ 0.60, acceptable ≥ 0.55)')
    print(f'nDCG@10:        {mean_ndcg:.4f}  (target ≥ 0.55, acceptable ≥ 0.50)')
    print(f'MRR status:     {"✅ PASS" if mrr >= 0.55 else "❌ FAIL"}')
    print(f'nDCG status:    {"✅ PASS" if mean_ndcg >= 0.50 else "❌ FAIL"}')
else:
    print('No relevance judgements available — cannot compute MRR/nDCG.')
    print('These metrics require data/evaluation/relevance_judgements.json with ground truth.')

mean_nepal = df['nepal_fraction'].mean()
mean_latency = df['latency_ms'].mean()
print(f'\nNepal fraction: {mean_nepal:.2%}  (target ≥ 0.60 for C4)')
print(f'Mean latency:   {mean_latency:.1f} ms per query')

print('\n--- Per-query results ---')
print(df[['label', 'n_retrieved', 'nepal_fraction', 'latency_ms', 'rr', 'ndcg']].to_string(index=False))

## 4. Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Nepal fraction per query
axes[0].bar(df['label'], df['nepal_fraction'], color='#4CAF50')
axes[0].axhline(0.6, color='red', linestyle='--', label='Target (0.60)')
axes[0].set_title('Nepal Citation Fraction per Query')
axes[0].set_ylabel('Fraction')
axes[0].set_ylim(0, 1)
axes[0].legend()
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=45, ha='right')

# Latency distribution
axes[1].hist(df['latency_ms'], bins=10, color='steelblue', edgecolor='white')
axes[1].axvline(df['latency_ms'].mean(), color='red', linestyle='--',
                label=f"Mean={df['latency_ms'].mean():.0f}ms")
axes[1].set_xlabel('Latency (ms)')
axes[1].set_ylabel('Count')
axes[1].set_title('Retrieval Latency')
axes[1].legend()

# RRF score distribution of retrieved docs (first query)
if results:
    first_docs = retriever.retrieve_raw(judgements[0]['query'], top_k=10)
    rrf_scores = [d.rrf_score for d in first_docs]
    axes[2].bar(range(len(rrf_scores)), rrf_scores, color='orange')
    axes[2].set_xlabel('Rank')
    axes[2].set_ylabel('RRF score')
    axes[2].set_title(f'RRF Scores — "{judgements[0]["label"]}"')

plt.tight_layout()
plt.savefig('../data/evaluation/c1_retrieval_benchmark.png', dpi=150, bbox_inches='tight')
plt.show()

# Save JSON results
import os; os.makedirs('../data/evaluation', exist_ok=True)
summary = {
    'mrr_at_10': float(evaluable['rr'].mean()) if len(evaluable) else None,
    'ndcg_at_10': float(evaluable['ndcg'].mean()) if len(evaluable) else None,
    'mean_nepal_fraction': float(df['nepal_fraction'].mean()),
    'mean_latency_ms': float(df['latency_ms'].mean()),
    'n_queries': len(df),
    'n_with_judgements': len(evaluable),
}
with open('../data/evaluation/c1_retrieval_quality.json', 'w') as f:
    json.dump(summary, f, indent=2)
print('Results saved to data/evaluation/c1_retrieval_quality.json')